In [3]:
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sys

root = os.path.abspath('../..')  
sys.path.append(root)

def plot_dgh_traces_with_breakpoints(
    path_dgh,
    path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    id_column='ID',
    bp1_column='breakpoint_1 (vp)',
    bp2_column='breakpoint_2 (vp)',
    title='Traces with breakpoints'
):
    """
    Lee un CSV de DGH con IDs y breakpoints, busca los archivos CSV de traces
    que empiezan con cada ID en otra carpeta, y los grafica de forma interactiva
    en Plotly con breakpoints superpuestos.

    Parámetros
    ----------
    path_dgh : str
        Ruta al CSV con columnas ID, breakpoint_1 (vp), breakpoint_2 (vp)
    path_sec_files : str
        Carpeta donde están los CSV de traces
    vp_column : str
        Nombre de la columna de posición vertical en los traces
    sec_column : str
        Nombre de la columna de conductividad corregida en los traces
    id_column : str
        Nombre de la columna ID en el CSV DGH
    bp1_column : str
        Nombre de la columna del primer breakpoint
    bp2_column : str
        Nombre de la columna del segundo breakpoint
    title : str
        Título de la figura

    Retorna
    -------
    fig : plotly.graph_objects.Figure
    """
    
    # -------------------------
    # Leer y validar archivo DGH
    # -------------------------
    df_dgh = pd.read_csv(path_dgh)
    required_dgh_cols = [id_column, bp1_column, bp2_column]
    missing_dgh = [c for c in required_dgh_cols if c not in df_dgh.columns]
    if missing_dgh:
        raise ValueError(f"Faltan columnas en path_dgh: {missing_dgh}")

    df_dgh = df_dgh[required_dgh_cols].copy()
    df_dgh[id_column] = df_dgh[id_column].astype(str)

    # -------------------------
    # Colores por ID
    # -------------------------
    unique_ids = df_dgh[id_column].dropna().unique().tolist()
    palette = px.colors.qualitative.Alphabet
    color_map = {id_val: palette[i % len(palette)] for i, id_val in enumerate(unique_ids)}
    
    fig = go.Figure()
    shown_legend_ids = set()

    # -------------------------
    # Iterar por cada ID
    # -------------------------
    for _, row in df_dgh.iterrows():
        id_val = str(row[id_column])
        bp_values = [row[bp1_column], row[bp2_column]]
        color = color_map[id_val]

        # Buscar archivos que empiecen con el ID
        pattern = os.path.join(path_sec_files, f"{id_val}*.csv")
        matched_files = sorted(glob.glob(pattern))

        if not matched_files:
            print(f"[AVISO] No se encontraron archivos para ID={id_val} con patrón: {pattern}")
            continue

        for file_path in matched_files:
            try:
                df_trace = pd.read_csv(file_path)
            except Exception as e:
                print(f"[AVISO] No se pudo leer {file_path}: {e}")
                continue

            # Validar columnas
            missing_trace = [c for c in [vp_column, sec_column] if c not in df_trace.columns]
            if missing_trace:
                print(f"[AVISO] En {os.path.basename(file_path)} faltan columnas: {missing_trace}")
                continue

            # Limpiar datos
            df_trace = df_trace[[vp_column, sec_column]].copy()
            df_trace[vp_column] = pd.to_numeric(df_trace[vp_column], errors='coerce')
            df_trace[sec_column] = pd.to_numeric(df_trace[sec_column], errors='coerce')
            df_trace = df_trace.dropna(subset=[vp_column, sec_column])

            if df_trace.empty:
                print(f"[AVISO] {os.path.basename(file_path)} quedó vacío después de limpiar NaNs.")
                continue

            trace_name = os.path.basename(file_path)

            # Mostrar leyenda solo una vez por ID en la línea principal
            showlegend_line = id_val not in shown_legend_ids

            fig.add_trace(
                go.Scatter(
                    x=df_trace[sec_column],
                    y=df_trace[vp_column],
                    mode='lines+markers',
                    name=f"{id_val}",
                    legendgroup=id_val,
                    showlegend=showlegend_line,
                    line=dict(color=color, width=0.8),
                    marker=dict(
                        color=color,
                        size=2,
                        symbol='circle'
                    ),
                    hovertemplate=(
                        f"ID: {id_val}<br>"
                        f"Archivo: {trace_name}<br>"
                        f"{sec_column}: %{{x}}<br>"
                        f"{vp_column}: %{{y}}<extra></extra>"
                    )
                )
            )

            shown_legend_ids.add(id_val)
               


    fig.update_layout(
        title=title,
        xaxis_title=sec_column,
        yaxis_title=vp_column,
        template='plotly_white',
        legend_title='ID',
        hovermode='closest',
        height=600
    )

    fig.update_yaxes(autorange='reversed')

    return fig

In [12]:
path_dgh = f'{root}/data/dgh_fwl_estimation_48000_raw_aw5.csv'
path_sec_files = f'{root}/data/raw/raw_aw5'

fig = plot_dgh_traces_with_breakpoints(
    path_dgh=path_dgh,
    path_sec_files=path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    title='AW5'
)

fig.show()